# 01 — CLIP Intuition (Beginner Notebook)

Welcome 👋 This notebook is for beginners (second Jupyter session).

## Learning goal
Understand in a concrete way what CLIP does:
- turn text and images into embeddings
- compare similarity between them
- observe why prompt wording matters

> Run each cell with **Shift + Enter**.


## Safety notes (read first)

- If a cell fails once: run it again.
- If imports fail: run the install cell, then retry imports.
- If model loading hangs: Runtime → Restart runtime, then run all from top.
- CPU works too, but may be slower than GPU.


## Roadmap

1. Setup
2. Imports + device check
3. Load CLIP model
4. Load one image
5. Compare prompt variants
6. Guided edits + reflection


## Mini glossary (quick)

- **Embedding**: a numeric vector representing meaning.
- **Logits**: raw matching scores from the model.
- **Softmax**: converts raw scores into normalized probabilities.
- **Similarity score**: how well text and image match in CLIP space.


## Step 0 — Setup (optional install)

**What this does:** installs required packages if your runtime does not have them.  
**Why it matters:** avoids import errors later.


In [ ]:
# If needed in Colab, uncomment and run once:
!pip -q install transformers torch pillow requests

print('Setup ready ✅')


## Step 1 — Imports and device check

**What this does:** imports libraries and checks if GPU is available.  
**Why it matters:** CLIP runs faster on GPU, but CPU is fine for learning.


In [ ]:
import torch
from PIL import Image
import requests
from io import BytesIO
from transformers import CLIPProcessor, CLIPModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)


✅ **Checkpoint:** You should see `Device: cuda` or `Device: cpu`.


## Step 2 — Load CLIP model

**What this does:** loads a pretrained CLIP model and its processor.  
**Why it matters:** the processor prepares text/images in the right format for the model.


In [ ]:
model_id = 'openai/clip-vit-base-patch32'
model = CLIPModel.from_pretrained(model_id).to(device)
processor = CLIPProcessor.from_pretrained(model_id)
print('Loaded:', model_id)


✅ **Checkpoint:** You should see `Loaded: openai/clip-vit-base-patch32`.


## Step 3 — Load one image

**What this does:** downloads a single image from URL and displays it.  
**Why it matters:** we need one fixed image to compare different text prompts fairly.


In [ ]:
image_url = 'https://images.unsplash.com/photo-1518791841217-8f162f1e1131?auto=format&fit=crop&w=900&q=80'
image = Image.open(BytesIO(requests.get(image_url, timeout=20).content)).convert('RGB')
image


✅ **Checkpoint:** The image should display correctly.


## Step 4 — Define prompt variants

**What this does:** creates three prompts with different style/detail levels.  
**Why it matters:** we can test how wording changes CLIP similarity.


In [ ]:
prompts = [
    'a cat',
    'a cyberpunk cat in neon lights',
    'a black and white portrait of a cat'
]
prompts


## Step 5 — Compute similarity scores

**What this does:** runs CLIP on the image + prompt list, then prints normalized scores.  
**Why it matters:** scores give evidence for which prompt best matches the image semantics.


In [ ]:
inputs = processor(text=prompts, images=image, return_tensors='pt', padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

probs = outputs.logits_per_image.softmax(dim=1).cpu().numpy()[0]
for p, s in zip(prompts, probs):
    print(f'{p:45s} -> {s:.4f}')


✅ **Checkpoint:** you should see 3 lines with prompt + score.

Interpretation hint: higher score = stronger CLIP match for this image.


## Guided exercise (beginner mode)

Edit exactly **one** prompt element, then rerun only the scoring cell.

Try these one by one:
- add one style word
- add one lighting word
- make one prompt more specific

Keep notes: which change increased/decreased score?


## Reflection (3–5 lines, with sentence starters)

- The highest-scoring prompt was __ because __.
- When I changed __, the score __.
- This suggests that in image generation, I should __.
